# CSV & SQLite → Knowledge Graph

How to ingest **structured data** (CSV files, SQLite databases) into a Knowledge Graph and query with Graph RAG.

**Prerequisites:** Neo4j running (`docker compose -f workshop/docker-compose.yml up -d`)

In [1]:
from pathlib import Path
import pandas as pd
import sqlite3
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv(Path(".").resolve().parent / ".env")
load_dotenv(Path(".").resolve() / ".env")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
NEO4J_URI = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", "workshop2024")

def ask_llm(prompt, system="You are a helpful assistant."):
    return llm.invoke([SystemMessage(content=system), HumanMessage(content=prompt)]).content

print("Setup complete!")

Setup complete!


---
## Delete All Existing Data in Neo4j

Always start clean. This deletes **everything** — all nodes and relationships.

In [2]:
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")
    count = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
print(f"Database cleared. Nodes remaining: {count}")
driver.close()

Database cleared. Nodes remaining: 0


---
## Part 1: CSV → Knowledge Graph

We'll use the AI companies CSV. Each row becomes **multiple nodes and relationships**.

In [3]:
df = pd.read_csv("../csv-to-kg/data/ai_companies.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}\n")
df.head()

Shape: (20, 10)
Columns: ['company', 'founded_year', 'headquarters', 'ceo', 'flagship_product', 'category', 'funding_total_b', 'investors', 'num_employees', 'key_technology']



,company,founded_year,headquarters,ceo,flagship_product,category,funding_total_b,investors,num_employees,key_technology
0,OpenAI,2015,San Francisco,Sam Altman,GPT-4,LLM Provider,11.3,Microsoft;Thrive Capital;Khosla Ventures,3500,Transformers
1,Anthropic,2021,San Francisco,Dario Amodei,Claude,LLM Provider,7.6,Google;Spark Capital;Salesforce Ventures,1200,Constitutional AI
2,Google DeepMind,2010,London,Demis Hassabis,Gemini,LLM Provider,0.0,Google,2500,Transformers
3,Meta AI,2013,Menlo Park,Yann LeCun,Llama,Open Source LLM,0.0,Meta,1500,Transformers
4,Mistral AI,2023,Paris,Arthur Mensch,Mixtral,Open Source LLM,2.2,Andreessen Horowitz;Lightspeed;Microsoft,600,Mixture of Experts


### How CSV rows become a graph

One flat CSV row:
```
company=OpenAI, ceo=Sam Altman, headquarters=San Francisco, investors=Microsoft;Thrive Capital
```

Becomes **multiple nodes + relationships**:
```
(Sam Altman:Person) --[LEADS]--> (OpenAI:Company)
(OpenAI:Company) --[HEADQUARTERED_IN]--> (San Francisco:City)
(Microsoft:Investor) --[INVESTED_IN]--> (OpenAI:Company)
(Thrive Capital:Investor) --[INVESTED_IN]--> (OpenAI:Company)
```

In [4]:
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)

with driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")
    
    for _, row in df.iterrows():
        # Company node
        s.run("""MERGE (c:Company {name: $n})
               SET c.founded=$y, c.funding_B=$f, c.employees=$e""",
              n=row["company"], y=int(row["founded_year"]),
              f=float(row["funding_total_b"]), e=int(row["num_employees"]))
        
        # CEO → LEADS → Company
        s.run("""MERGE (p:Person {name: $ceo})
               MERGE (c:Company {name: $co})
               MERGE (p)-[:LEADS]->(c)""",
              ceo=row["ceo"], co=row["company"])
        
        # Company → HEADQUARTERED_IN → City
        s.run("""MERGE (ci:City {name: $city})
               MERGE (c:Company {name: $co})
               MERGE (c)-[:HEADQUARTERED_IN]->(ci)""",
              city=row["headquarters"], co=row["company"])
        
        # Company → PRODUCES → Product
        s.run("""MERGE (pr:Product {name: $prod})
               MERGE (c:Company {name: $co})
               MERGE (c)-[:PRODUCES]->(pr)""",
              prod=row["flagship_product"], co=row["company"])
        
        # Company → IN_CATEGORY → Category
        s.run("""MERGE (cat:Category {name: $cat})
               MERGE (c:Company {name: $co})
               MERGE (c)-[:IN_CATEGORY]->(cat)""",
              cat=row["category"], co=row["company"])
        
        # Company → USES_TECH → Technology
        s.run("""MERGE (t:Technology {name: $tech})
               MERGE (c:Company {name: $co})
               MERGE (c)-[:USES_TECH]->(t)""",
              tech=row["key_technology"], co=row["company"])
        
        # Investors → INVESTED_IN → Company
        for inv in str(row["investors"]).split(";"):
            inv = inv.strip()
            if inv:
                s.run("""MERGE (i:Investor {name: $inv})
                       MERGE (c:Company {name: $co})
                       MERGE (i)-[:INVESTED_IN]->(c)""",
                      inv=inv, co=row["company"])
        
        print(f"  Loaded: {row['company']}")
    
    nodes = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    edges = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    labels = s.run("CALL db.labels() YIELD label RETURN collect(label) AS l").single()["l"]

print(f"\nGraph: {nodes} nodes, {edges} relationships")
print(f"Node types: {labels}")
driver.close()

  Loaded: OpenAI
  Loaded: Anthropic
  Loaded: Google DeepMind
  Loaded: Meta AI
  Loaded: Mistral AI
  Loaded: Cohere
  Loaded: LangChain
  Loaded: Pinecone
  Loaded: Weaviate
  Loaded: ChromaDB
  Loaded: Neo4j
  Loaded: Hugging Face
  Loaded: Stability AI
  Loaded: Perplexity AI
  Loaded: Cursor
  Loaded: Replit
  Loaded: Adept AI
  Loaded: Jasper AI
  Loaded: Scale AI
  Loaded: Databricks

Graph: 128 nodes, 151 relationships
Node types: ['Person', 'Company', 'City', 'Product', 'Category', 'Technology', 'Investor']


### Visualize the CSV-based graph

In [5]:
from pyvis.network import Network

COLORS = {
    "Company": "#4ECDC4", "Person": "#FF6B6B", "City": "#FFEAA7",
    "Product": "#45B7D1", "Category": "#96CEB4", "Technology": "#DDA0DD",
    "Investor": "#F39C12",
}

net = Network(height="800px", width="100%", directed=True, bgcolor="#1a1a2e",
              font_color="white", cdn_resources="remote")
net.barnes_hut(gravity=-5000, spring_length=200)

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    for r in s.run("MATCH (n) RETURN n.name AS name, labels(n)[0] AS label"):
        color = COLORS.get(r["label"], "#DFE6E9")
        size = 25 if r["label"] in ("Company", "Investor") else 15
        net.add_node(r["name"], label=r["name"], color=color, size=size,
                     title=r["label"])
    for r in s.run("MATCH (a)-[r]->(b) RETURN a.name AS s, b.name AS t, type(r) AS rel"):
        net.add_edge(r["s"], r["t"], label=r["rel"], color="#636e72",
                     font={"size": 8, "color": "#b2bec3"})
driver.close()

html_path = os.path.abspath("csv_kg.html")
net.save_graph(html_path)
os.system(f"open '{html_path}'")
print(f"Opened: {html_path}")

Opened: /Users/datasense/Desktop/knowledgegraphs/workshop/csv_kg.html


### Graph RAG over CSV data

In [6]:
def graph_rag_csv(question):
    driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
    with driver.session() as s:
        labels = s.run("CALL db.labels() YIELD label RETURN collect(label) AS l").single()["l"]
        rel_types = s.run("CALL db.relationshipTypes() YIELD relationshipType RETURN collect(relationshipType) AS r").single()["r"]
        sample = [dict(r) for r in s.run("MATCH (n) RETURN n.name AS name, labels(n)[0] AS type LIMIT 30")]
    
    cypher = ask_llm(
        prompt=f"Convert to Cypher:\n\n{question}",
        system=f"""Neo4j Cypher expert. Schema:
Node labels: {labels}
Relationship types: {rel_types}
Sample entities: {sample}
Properties on Company: name, founded, funding_B, employees
Properties on other nodes: name
Return ONLY Cypher, no markdown."""
    ).strip().replace("```cypher", "").replace("```", "").strip()
    
    print(f"Cypher: {cypher}\n")
    
    try:
        with driver.session() as s:
            rows = [dict(r) for r in s.run(cypher)]
    except:
        with driver.session() as s:
            rows = [dict(r) for r in s.run(
                "MATCH (a)-[r]->(b) RETURN a.name AS source, type(r) AS rel, b.name AS target LIMIT 30"
            )]
    
    driver.close()
    context = "\n".join([str(r) for r in rows])
    return ask_llm(
        f"Graph results:\n{context}\n\nQuestion: {question}",
        system="Answer using ONLY graph results. Be specific."
    )

print("graph_rag_csv() ready!")

graph_rag_csv() ready!


In [7]:
print(graph_rag_csv("Which investor has funded the most companies? List them."))

Cypher: MATCH (i:Investor)-[:INVESTED_IN]->(c:Company)
WITH i, COUNT(c) AS fundedCompanies
ORDER BY fundedCompanies DESC
RETURN i.name AS Investor, fundedCompanies
LIMIT 10

The investor that has funded the most companies is Andreessen Horowitz, with 5 funded companies.


In [ ]:
print(graph_rag_csv("What technologies are used by companies in San Francisco?"))

In [ ]:
print(graph_rag_csv("Recommend companies similar to OpenAI based on shared investors and technology."))

---
## Part 2: SQLite → Knowledge Graph

Same concept but from a database. We'll create a small SQLite DB, then load it into Neo4j.

In [ ]:
# Create a sample SQLite database
conn = sqlite3.connect(":memory:")

conn.execute("""CREATE TABLE employees (
    id INTEGER PRIMARY KEY, name TEXT, role TEXT, department TEXT,
    manager_id INTEGER, city TEXT, skill TEXT, hobby TEXT
)""")

employees = [
    (1, "Alice Johnson", "CEO", "Executive", None, "New York", "Leadership", "Golf"),
    (2, "Bob Smith", "CTO", "Engineering", 1, "San Francisco", "Python", "Hiking"),
    (3, "Carol Davis", "VP Sales", "Sales", 1, "Chicago", "Negotiation", "Tennis"),
    (4, "Dave Wilson", "Senior Engineer", "Engineering", 2, "San Francisco", "Machine Learning", "Chess"),
    (5, "Eve Martinez", "Data Scientist", "Engineering", 2, "Austin", "Python", "Painting"),
    (6, "Frank Lee", "Sales Manager", "Sales", 3, "Chicago", "CRM", "Running"),
    (7, "Grace Kim", "ML Engineer", "Engineering", 4, "San Francisco", "PyTorch", "Yoga"),
    (8, "Henry Brown", "Account Exec", "Sales", 6, "New York", "Negotiation", "Cooking"),
    (9, "Ivy Chen", "DevOps", "Engineering", 2, "Austin", "Kubernetes", "Gaming"),
    (10, "Jack Taylor", "Designer", "Product", 1, "New York", "Figma", "Photography"),
]

conn.executemany("INSERT INTO employees VALUES (?,?,?,?,?,?,?,?)", employees)
conn.commit()

df_sql = pd.read_sql("SELECT * FROM employees", conn)
print(f"SQLite table: {len(df_sql)} employees\n")
df_sql

In [ ]:
# Load SQLite → Neo4j
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)

with driver.session() as s:
    s.run("MATCH (n) DETACH DELETE n")
    
    for _, row in df_sql.iterrows():
        # Person node
        s.run("MERGE (p:Person {name: $name}) SET p.role = $role",
              name=row["name"], role=row["role"])
        
        # Department
        s.run("""MERGE (d:Department {name: $dept})
               MERGE (p:Person {name: $name})
               MERGE (p)-[:IN_DEPARTMENT]->(d)""",
              dept=row["department"], name=row["name"])
        
        # City
        s.run("""MERGE (c:City {name: $city})
               MERGE (p:Person {name: $name})
               MERGE (p)-[:BASED_IN]->(c)""",
              city=row["city"], name=row["name"])
        
        # Skill
        s.run("""MERGE (sk:Skill {name: $skill})
               MERGE (p:Person {name: $name})
               MERGE (p)-[:HAS_SKILL]->(sk)""",
              skill=row["skill"], name=row["name"])
        
        # Hobby
        s.run("""MERGE (h:Hobby {name: $hobby})
               MERGE (p:Person {name: $name})
               MERGE (p)-[:HAS_HOBBY]->(h)""",
              hobby=row["hobby"], name=row["name"])
    
    # Manager relationships (from manager_id)
    for _, row in df_sql.iterrows():
        if pd.notna(row["manager_id"]):
            manager = df_sql[df_sql["id"] == int(row["manager_id"])].iloc[0]
            s.run("""MATCH (p:Person {name: $name}), (m:Person {name: $mgr})
                   MERGE (p)-[:REPORTS_TO]->(m)""",
                  name=row["name"], mgr=manager["name"])
    
    nodes = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    edges = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]

print(f"Graph: {nodes} nodes, {edges} relationships")
print(f"\nNeo4j: MATCH (n)-[r]->(m) RETURN n, r, m")
driver.close()
conn.close()

In [ ]:
# Visualize
from pyvis.network import Network

COLORS = {"Person": "#FF6B6B", "Department": "#4ECDC4", "City": "#FFEAA7",
          "Skill": "#DDA0DD", "Hobby": "#FF69B4"}

net = Network(height="700px", width="100%", directed=True, bgcolor="#1a1a2e",
              font_color="white", cdn_resources="remote")
net.barnes_hut(gravity=-4000)

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:
    for r in s.run("MATCH (n) RETURN n.name AS name, labels(n)[0] AS label, n.role AS role"):
        color = COLORS.get(r["label"], "#DFE6E9")
        title = f"{r['label']}: {r['name']}"
        if r["role"]: title += f" ({r['role']})"
        net.add_node(r["name"], label=r["name"], color=color, title=title,
                     size=20 if r["label"] == "Person" else 12)
    for r in s.run("MATCH (a)-[r]->(b) RETURN a.name AS s, b.name AS t, type(r) AS rel"):
        color = "#e17055" if r["rel"] == "REPORTS_TO" else "#636e72"
        net.add_edge(r["s"], r["t"], label=r["rel"], color=color,
                     font={"size": 8, "color": "#b2bec3"})
driver.close()

html_path = os.path.abspath("sqlite_kg.html")
net.save_graph(html_path)
os.system(f"open '{html_path}'")
print(f"Opened: {html_path}")

In [ ]:
# Graph RAG over SQLite data
print(graph_rag_csv("Who reports to Bob Smith and what are their skills?"))

In [ ]:
print(graph_rag_csv("Which employees share the same hobby?"))

In [ ]:
print(graph_rag_csv("What is the reporting chain from Grace Kim to Alice Johnson?"))

---
## Key Takeaway

**Any structured data → Knowledge Graph:**
- CSV: each row → multiple nodes + relationships (use MERGE to avoid duplicates)
- SQLite: same pattern, just read with `pd.read_sql()` first
- The graph reveals connections invisible in flat tables: shared investors, reporting chains, skill overlaps